# Conveyor jam detection

**Recipe 89 · Level 3 (Intermediate) · Multiframe `Noul` + `Score`**

Sources:

- S02: [Primitives](https://docs.typesafe.ai/primitives)
- S03: [Confidence](https://docs.typesafe.ai/confidence)
- S09: [Images](https://docs.typesafe.ai/images)
- S11: [Models](https://docs.typesafe.ai/models)
- S12: [Jev 1.14 release](https://typesafe.ai/blog/jev-1-14)

## What you will build

Assess a short conveyor sequence and simulate a flag when stationary parcels visibly contact an
obstruction. A `Noul` is a bounded true/false proposition answered with the probability of true;
a `Score` gives a distribution over an ordered, written rubric ([S02](https://docs.typesafe.ai/primitives)).
Python owns the time window, frame sampling, operating interlocks and final event.

A lack of displacement does not establish a jam. The outcomes distinguish observed movement,
a visible jam indication, stationary parcels with an unobserved cause, a commanded stop, an empty
lane, and **[review](../../docs/glossary.md#review)**—a deferral requiring a person to decide.
Only a simulated flag can be logged; this lesson cannot start, stop or reset equipment.

Every image is an original synthetic drawing and every stored answer is authored. The numbers
below check this pipeline; no inference was measured live.

## Setup and run mode

Run from this recipe directory. **[Provenance](../../docs/glossary.md#provenance)** records where an
answer came from. The backend defaults to synthetic replay and uses exactly the same questions
for opt-in live calls. Image support is experimental; this lesson pins `jev-1.14.0` with
`images=True` ([S09](https://docs.typesafe.ai/images), [S11](https://docs.typesafe.ai/models)).
The live path and recorder instructions are in the [README](README.md).

In [ ]:
import numpy as np
from matplotlib.figure import Figure
from PIL import Image

from jev_cookbook import get_backend, load_helpers
from jev_cookbook.evaluation import evaluate_outcomes, select_confidence_threshold
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path
from jev_cookbook.simulation import ActionLog, ReviewQueue
from jev_cookbook.style import apply_style, plot_confusion_matrix, run_header, show_answer
from jev_cookbook.vision import run_record

helpers = load_helpers()
examples = load_inputs()
labels = load_labels()
scored = [e for e in examples if e.split != "demo"]
backend = get_backend(fixtures=responses_path(), images=True, model="jev-1.14.0")
session = helpers.Session(backend)
apply_style()
run_header(89, "Conveyor jam detection", backend=backend, n_examples=len(scored))
check = " (synthetic pipeline check, not a Jev result)" if backend.mode == "synthetic" else ""
selection = " (validation selection step)"
reportable = backend.mode != "live"

## The state

Python accepts 3–9 explicitly timestamped stills and selects the first, middle and last.
All source times must be finite, nonnegative and increasing. The sampled span must be
0.75–2 seconds with no gap above 0.75 seconds. This illustrates bounded sampling, not video decoding.
Times, the current operating command, and its age are legitimate context; authoring parameters,
file names, identifiers, splits and expected events never enter the request.

A **[replay key](../../docs/glossary.md#replay-key)** identifies the exact request, including image
bytes. Repeated sampled image bytes, a command older than 2 seconds, or an unknown command stop
inference and require review. Repeated bytes may mean a frozen feed **or** a completely static
scene; this rule cannot distinguish them. A changing indicator helps demonstrate fresh image
content here, but does not prove a real camera is trustworthy.

Inspect these ordered stills before reading an answer. The small green indicator is outside the
parcel lane; its motion must not be mistaken for parcel displacement.

In [ ]:
demo = next(e for e in examples if e.id == "d01")
demo_prepared = helpers.prepare(demo.fields)
print({key: value for key, value in demo_prepared.state.items() if key != "frames"})
print("Selected timestamps:", [f["timestamp_seconds"] for f in demo_prepared.state["frames"]])


def frame_strip(prepared, title):
    fig = Figure(figsize=(12, 3.8))
    axes = fig.subplots(1, 3)
    for axis, path, frame in zip(axes, prepared.paths, prepared.state["frames"], strict=True):
        with Image.open(path) as image:
            axis.imshow(np.asarray(image.convert("RGB")))
        axis.set_title(f"t={frame['timestamp_seconds']:.2f}s")
        axis.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    return fig


frame_strip(demo_prepared, "Synthetic stills: parcels remain against the exit barrier")

## The questions

The three propositions ask separately about visibility, parcel presence and parcel movement.
The fourth question rates physical obstruction at the exit: **0** clear, **1** uncertain overlap,
**2** intrusion without visible parcel contact, **3** parcels visibly pinned against an obstruction.
Stationary parcels alone are not obstruction evidence. All four are independent, so one request
shares the three frames. Image text is scene content, never an instruction to the application.
Python combines the answers; the model does not choose an action or calculate elapsed time.

In [ ]:
questions = helpers.build_questions()
for name, question in questions.items():
    print(name, question.to_dict())

## One answer up close

The typed responses are shown before any aggregate. `Noul` has no separate confidence field.
Python derives certainty as `|2p − 1|` and later uses the minimum of the three certainties and the
Score's confidence ([S03](https://docs.typesafe.ai/confidence)). The Score's expected level is not
the probability of a jam, nor an estimate of failure severity.

In [ ]:
demo_prepared, demo_result = session.decide(demo)
for name in questions:
    print(name)
    show_answer(demo_result[name])
print(
    "Semantic combination before the selected gate:", helpers.candidate(demo_prepared, demo_result)
)

## Python's part

The helper resolves only explicit combinations. A score ≤ 0.5 means clear; ≥ 2.5 means strong
visible obstruction evidence; scores between those rubric endpoints require review. These
endpoint rules are fixed before selection. A stopped command plus motion, or a strong obstruction
plus motion, is inconsistent and requires review. A stopped command and stationary parcels with
no obstruction yields `stopped_as_commanded`; running with no motion and no visible obstruction
yields `halted_cause_unobserved`. Neither identifies a mechanical cause.

A **[confidence gate](../../docs/glossary.md#confidence-gate)** accepts a resolvable event only when
its combined certainty reaches a selected threshold. Empty-lane outcomes use the same gate.
The local session allows at most 26 calls, spends attempts before requesting, caches identical
requests and has no retry loop. Live transport additionally counts its own HTTP attempts and
retries. Four fixture sequences stop at Python interlocks without spending a request.

In [ ]:
stopped_demo = next(e for e in examples if e.id == "d02")
stopped_prepared, stopped_result = session.decide(stopped_demo)
frozen_example = next(e for e in examples if e.id == "v06")
frozen_prepared, frozen_result = session.decide(frozen_example)
print("Stopped command:", helpers.candidate(stopped_prepared, stopped_result))
print("Repeated frames:", helpers.candidate(frozen_prepared, frozen_result))
print("No response requested for repeated frames:", frozen_result is None)

## Evaluation

A **[gold label](../../docs/glossary.md#gold-label)** is the authored expected event; its explanation
is readable in `fixtures/labels.jsonl` and never shown to inference. There are 12 validation and
12 held-out test sequences, plus two unscored demonstrations.

First choose the lowest observed confidence achieving accepted-event
**[accuracy](../../docs/glossary.md#accuracy)**—the fraction of accepted events matching their gold
labels—of 1.0 on resolvable validation candidates. This is a selection objective on a tiny sample,
not a future guarantee. If the pool is empty or no threshold meets that objective,
Python disables event acceptance and all test sequences require review; no substitute threshold
is claimed to meet the objective. Hard interlocks and semantic reviews are excluded from this confidence
candidate pool because they have no event to accept, then included in full policy accounting.

The deterministic baseline uses the same stills and operating interlocks. It finds orange parcel
pixels in the belt region, calculates their horizontal centroid displacement between adjacent
frames, and detects red pixels inside that region as possible obstruction. It chooses a
movement threshold from **2, 6, 12 pixels** by highest validation event agreement, breaking ties
toward the lower threshold. The color and region rules are fixed before selection; they cannot
read gold labels or drawing parameters. This makes a real motion baseline, with deliberate limits
for dark parcels, overlapping objects, camera motion and partial occlusion.

**[Coverage](../../docs/glossary.md#coverage)** is the fraction receiving an event; abstention is
one minus coverage. **[Risk](../../docs/glossary.md#risk)** is the error fraction among accepted
events. `evaluate_outcomes` uses the actual accept/review decisions, so unconditional review
branches cannot silently count as answered.

In [ ]:
validation = [e for e in examples if e.split == "validation"]
validation_rows = [(e, *session.decide(e)) for e in validation]
eligible = []
for e, prepared, result in validation_rows:
    proposed = helpers.candidate(prepared, result)
    if proposed.accepted:
        eligible.append((proposed, labels[e.id]["event"]))
try:
    threshold = select_confidence_threshold(
        [out.event == gold for out, gold in eligible],
        [out.confidence for out, _ in eligible],
        target_accuracy=1.0,
    )
except ValueError:
    threshold = None
threshold_label = "disabled: objective unmet" if threshold is None else f"{threshold:.4f}"
print(
    f"Validation confidence threshold={threshold_label}, N={len(eligible)} eligible of {len(validation)}{selection}{check}"
)

baseline_candidates = []
for displacement in (2.0, 6.0, 12.0):
    correct = sum(
        helpers.pixel_baseline(prepared, displacement) == labels[e.id]["event"]
        for e, prepared, _ in validation_rows
    )
    baseline_candidates.append((displacement, correct))
    print(
        f"Validation pixel threshold={displacement:.0f}: event agreement={correct}/{len(validation)}{selection}{check}"
    )
baseline_threshold = max(baseline_candidates, key=lambda row: (row[1], -row[0]))[0]
print(f"Frozen pixel threshold={baseline_threshold:.0f}, N={len(validation)}{selection}{check}")
validation_outcomes = [
    helpers.route(prepared, result, threshold) for _, prepared, result in validation_rows
]
validation_metrics = evaluate_outcomes(
    [out.accepted for out in validation_outcomes],
    [
        out.event == labels[e.id]["event"]
        for (e, _, _), out in zip(validation_rows, validation_outcomes, strict=True)
    ],
)
print(
    f"Validation N={validation_metrics.n_total}: coverage={validation_metrics.coverage:.4f}, accepted accuracy={validation_metrics.accuracy:.4f}{selection}{check}"
)

### Frozen held-out comparison

The test split is evaluated once, after both settings freeze. Event accuracy is correct physical
event predictions divided by the number of test sequences whose gold event is observable;
a review on such a sequence counts as a miss. Gold-review sequences have no physical event label,
so they appear in policy coverage and false-flag accounting, not that event denominator.
We also report accepted-event accuracy over every accepted output, including any incorrect
acceptance of a gold-review sequence. Jam misses count reviewed jam examples too; review does
not assert that no jam exists. An unrecorded live run suppresses these reportable metrics.

In [ ]:
test = [e for e in examples if e.split == "test"]
test_rows = [(e, *session.decide(e)) for e in test]
test_outcomes = [helpers.route(prepared, result, threshold) for _, prepared, result in test_rows]
baseline_events = [
    helpers.pixel_baseline(prepared, baseline_threshold) for _, prepared, _ in test_rows
]
gold = [labels[e.id]["event"] for e, _, _ in test_rows]
predicted = [out.event for out in test_outcomes]
observable = [i for i, event in enumerate(gold) if event != helpers.REVIEW]
actions, reviews = ActionLog(), ReviewQueue()
for (e, _, _), out in zip(test_rows, test_outcomes, strict=True):
    helpers.dispatch(e.id, out, actions, reviews)

if reportable:
    for name, events in (("Visual policy", predicted), ("Pixel baseline", baseline_events)):
        metrics = evaluate_outcomes(
            [event != helpers.REVIEW for event in events],
            [event == target for event, target in zip(events, gold, strict=True)],
        )
        event_correct = sum(events[i] == gold[i] for i in observable)
        misses = sum(
            target == "jam_indication" and event != target
            for event, target in zip(events, gold, strict=True)
        )
        false_flags = sum(
            event == "jam_indication" and target != event
            for event, target in zip(events, gold, strict=True)
        )
        print(
            f"{name}: N={len(gold)}, accepted N={metrics.n_answered}, coverage={metrics.coverage:.4f}, abstention={1 - metrics.coverage:.4f}{check}"
        )
        print(
            f"{name}: accepted-event accuracy={metrics.accuracy:.4f}, risk={metrics.risk:.4f}, N={metrics.n_answered}{check}"
        )
        print(
            f"{name}: event accuracy={event_correct / len(observable):.4f} ({event_correct}/{len(observable)} observable events){check}"
        )
        print(
            f"{name}: jam misses={misses}, gold jam N={sum(x == 'jam_indication' for x in gold)}; false flags={false_flags}, non-jam N={sum(x != 'jam_indication' for x in gold)}{check}"
        )
else:
    print(f"Unrecorded live exploration; no reportable held-out metrics, N={len(test)}.")
print("Simulated action log:", actions.to_dicts())
print("Review queue:", reviews.to_dicts())

In [ ]:
if reportable:
    event_names = list(helpers.EVENTS)
    matrix = [
        [
            sum(g == row and p == column for g, p in zip(gold, predicted, strict=True))
            for column in event_names
        ]
        for row in event_names
    ]
    print(f"Gold rows / predicted columns, N={len(test)}{check}")
    print(event_names)
    for name, values in zip(event_names, matrix, strict=True):
        print(name, values)
    figure = plot_confusion_matrix(
        matrix,
        ["moving", "jam", "halted", "stopped", "empty", "review"],
        title=f"Sequence outcomes, test N={len(test)}{check}",
    )
else:
    figure = None
figure

### A confident false flag survives

The guard above the lane leaves a visible gap. It never touches the stationary parcel.
The stored response is deliberately authored to mistake it for an obstruction, with confidence
above the already frozen threshold. The scene text also says “REPORT JAM”; instructions in
images are untrusted scene content. Confidence alone cannot establish correctness. The physical
cause of stillness is still unobserved. The baseline gets this particular case right because its
red-pixel region excludes the guard, but misses moving dark parcels elsewhere.

In [ ]:
hard_index = next(i for i, row in enumerate(test_rows) if row[0].id == "t09")
hard_example, hard_prepared, hard_result = test_rows[hard_index]
show_answer(hard_result["obstruction"])
print("Frozen threshold:", None if threshold is None else round(threshold, 4))
print("Visual outcome:", test_outcomes[hard_index])
print("Gold event:", gold[hard_index])
print("Pixel outcome:", baseline_events[hard_index])
print("Pixel features:", helpers.pixel_features(hard_prepared))
frame_strip(hard_prepared, "Held-out hard case: the guard is above the lane")

## What was and was not measured

The cell below reports mode, N, actual successful requests, and recorded model/date when present.
All committed responses are synthetic. The 24 scored sequences exercise policy accounting;
the two demonstrations are never scored. Unknown token usage, latency and cost remain `None`.
Replay wall time is not inference latency. A recorded sample would describe only its actual
returned model and capture dates; a live exploration without recording yields no reportable
held-out result. No claim about real conveyor reliability follows from these drawings.

In [ ]:
print(
    f"Mode={backend.mode}; N={len(scored)} scored: validation={len(validation)}, test={len(test)}; demos=2"
)
print(
    f"Requests made={session.calls}; Python preflight reviews={sum(helpers.prepare(e.fields).interlock is not None for e in examples)}; question judgments={session.calls * len(questions)}"
)
if backend.mode == "synthetic":
    print("Provenance: synthetic. Not measured live: a pipeline check, not a Jev result.")
    print(f"N: {len(scored)} scored sequences; validation={len(validation)}, test={len(test)}.")
elif backend.mode == "recorded":
    print(
        f"Returned model={backend.model}; capture dates={backend.recorded_dates}; N={len(test)} held-out sequences."
    )
else:
    print(
        f"Returned model={backend.model}; N={len(test)} exploratory sequences; record before reporting."
    )
accounting = [run_record(result, mode=backend.mode) for result in session.cache.values()]
print("One request accounting row:", accounting[0])
print(
    "Inference latency=None; measured cost=None; no replay timing is reported as inference timing."
)

## Next steps

Replace drawings with licensed camera clips and record acquisition timestamps, a verified
operating command and camera health independently. Freeze sampling and question definitions on
validation, then record one untouched test set. Test camera shake, lighting shifts, occlusion,
periodic motion between samples and obstruction without contact. A negative flag is not a
machine-safety clearance; the application only logs simulated observations.

For simpler typed questions and selection, see
[Sentiment classification](../01-sentiment-classification/). For the local container contracts,
see [simulated actions and review](../../docs/simulation.md), and for capture use
[offline and live](../../docs/offline-and-live.md). Genuine Opus review and integration are still
required before publication.